# 大涨大跌六位小数一致性诊断

本 Notebook 只读取 `runtime_outputs/` 中已经生成的对比结果，不会重跑模型、重建候选或修改任何输出。

最终判断以预测信号和收益为主：预测标签、实际极端标签、命中、方向、O2O 和方向化 O2O。`score` 六位小数只作为辅助诊断。

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd

def find_package_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    candidates.extend([
        Path('/home/hzy/cta/05_上传包_日期解析修正版'),
        Path('/hpfs/innofs/home/hzy/cta/05_上传包_日期解析修正版'),
    ])
    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'src' / 'remote_validation.py').is_file() and (candidate / 'runtime_outputs').is_dir():
            return candidate
    raise FileNotFoundError('没有找到上传包根目录，请确认 Notebook 与 runtime_outputs 位于同一个上传包内。')

PACKAGE_ROOT = find_package_root(Path.cwd())
OUTPUT_DIR = Path(os.environ.get('UPLOAD_OUTPUT_DIR', str(PACKAGE_ROOT / 'runtime_outputs'))).expanduser().resolve()
MANIFEST_PATH = OUTPUT_DIR / '最终一致性结论.json'
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'没有找到 {MANIFEST_PATH}，请先运行主入口。')

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
print('结果目录：', OUTPUT_DIR)
print('原始 manifest success：', manifest.get('success'))

def load_compare(side: str) -> pd.DataFrame:
    path = OUTPUT_DIR / f'大涨大跌_{side}_逐日对比.csv'
    if not path.is_file():
        raise FileNotFoundError(f'没有找到逐日对比文件：{path}')
    return pd.read_csv(path, encoding='utf-8-sig')

def as_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype('string').str.strip().str.lower().isin(['true', '1', 'yes'])

def six_decimal_text(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors='coerce')
    return numeric.map(lambda value: '<NA>' if pd.isna(value) else f'{float(value):.6f}')

def six_decimal_equal(left: pd.Series, right: pd.Series) -> pd.Series:
    return six_decimal_text(left).eq(six_decimal_text(right))

signal_fields = ['predicted', 'actual_extreme', 'correct', 'direction_correct']
return_fields = ['o2o_bp', 'signed_o2o_bp']
numeric_result_fields = ['o2o_bp', 'signed_o2o_bp']
summary_rows = []
detail_tables = {}
example_tables = {}

for side, label in [('down', '大跌'), ('up', '大涨')]:
    frame = load_compare(side)
    common = frame['_merge'].astype('string').eq('both')
    raw_score_match = as_bool(frame['match_score'])
    score_6dp_match = six_decimal_equal(frame['score_generated'], frame['score_local'])

    mismatch_counts = {
        field: int((common & ~as_bool(frame[f'match_{field}'])).sum())
        for field in signal_fields + return_fields + ['phase']
    }
    signal_mismatch = sum(mismatch_counts[field] for field in signal_fields)
    return_mismatch = sum(mismatch_counts[field] for field in return_fields)

    result_rows = []
    for field in ['score'] + numeric_result_fields:
        if field == 'score':
            equal = score_6dp_match
        else:
            equal = six_decimal_equal(frame[f'{field}_generated'], frame[f'{field}_local'])
        result_rows.append({
            '字段': field,
            '六位小数不一致': int((common & ~equal).sum()),
            '六位小数一致': int((common & equal).sum()),
        })
    detail_tables[side] = pd.DataFrame(result_rows)

    raw_diff = (
        pd.to_numeric(frame['score_generated'], errors='coerce')
        - pd.to_numeric(frame['score_local'], errors='coerce')
    ).abs()
    common_diff = raw_diff.loc[common]
    summary_rows.append({
        '方向': label,
        '共同日期数': int(common.sum()),
        '原始 score 不一致': int((common & ~raw_score_match).sum()),
        'score 六位小数不一致': int((common & ~score_6dp_match).sum()),
        'score 六位小数全部一致': bool(len(common_diff) > 0 and score_6dp_match.loc[common].all()),
        '预测信号不一致': mismatch_counts['predicted'],
        '实际极端标签不一致': mismatch_counts['actual_extreme'],
        '命中标记不一致': mismatch_counts['correct'],
        '方向标记不一致': mismatch_counts['direction_correct'],
        'O2O 不一致': mismatch_counts['o2o_bp'],
        '方向化 O2O 不一致': mismatch_counts['signed_o2o_bp'],
        '阶段不一致（辅助）': mismatch_counts['phase'],
        '最大 score 绝对差': float(common_diff.max()) if len(common_diff) else np.nan,
        '平均 score 绝对差': float(common_diff.mean()) if len(common_diff) else np.nan,
        '信号和收益口径一致': bool(signal_mismatch == 0 and return_mismatch == 0),
    })

    bad_6dp = frame.loc[common & ~score_6dp_match].copy()
    if bad_6dp.empty:
        example_tables[side] = pd.DataFrame(columns=['日期', '远端 score', '本地 score', '远端六位小数', '本地六位小数'])
    else:
        examples = bad_6dp.head(10).copy()
        example_tables[side] = pd.DataFrame({
            '日期': examples['date'],
            '远端 score': examples['score_generated'],
            '本地 score': examples['score_local'],
            '远端六位小数': six_decimal_text(examples['score_generated']),
            '本地六位小数': six_decimal_text(examples['score_local']),
        })

print('===== 最重要：信号和收益口径结论（优先看这一张） =====')
display(pd.DataFrame(summary_rows))

In [ ]:
for side, label in [('down', '大跌'), ('up', '大涨')]:
    print(f'===== {label}：数值字段六位小数检查 =====')
    display(detail_tables[side])
    print(f'===== {label}：六位小数仍不一致的示例 =====')
    display(example_tables[side])

print('请截图顺序：先截“信号和收益口径结论”，再截大跌和大涨的数值字段检查。')
print('最终主要看：预测信号、实际极端标签、命中、方向、O2O 和方向化 O2O 的不一致数是否全部为 0。')
print('score 六位小数不一致为 0 是辅助加分项，不应单独覆盖业务信号和收益结论。')